# Original ST-GCN (yysijie/st-gcn) on Gym99
Clones the original ST-GCN repo, feeds Gym99 data through it, and trains.

**Data note**: original ST-GCN expects `(N, 3, T, V, M)` with OpenPose 18-joint ordering.
Our Gym99 pipeline produces `(N, 2, T, 18, 1)` in COCO17 ordering.  
We add a dummy confidence channel (ones) to match 3 channels.  
Joint ordering differs from OpenPose — graph edges are approximate for this test.

In [ ]:
# ── Cell 1: Clone repos ───────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

OUR_REPO_URL  = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
ORIG_REPO_URL = 'https://github.com/yysijie/st-gcn.git'
BRANCH        = 'refactor-1'

OUR_DIR  = Path('/content/Yolo-ST-GCN')
ORIG_DIR = Path('/content/st-gcn')

# Our repo
if not OUR_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch',
                    OUR_REPO_URL, str(OUR_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(OUR_DIR), 'pull', 'origin', BRANCH], check=True)

# Original ST-GCN repo
if not ORIG_DIR.exists():
    subprocess.run(['git', 'clone', ORIG_REPO_URL, str(ORIG_DIR)], check=True)

print('Our repo   :', OUR_DIR)
print('Orig ST-GCN:', ORIG_DIR)

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub', 'scipy', 'scikit-learn', 'tqdm',
                'pyyaml', 'tensorboardX'], check=True)

# Install torchlight from original repo
torchlight_dir = ORIG_DIR / 'torchlight'
result = subprocess.run(
    [sys.executable, 'setup.py', 'install'],
    cwd=str(torchlight_dir),
    capture_output=True, text=True
)
if result.returncode == 0:
    print('torchlight installed.')
else:
    print('torchlight install warning:', result.stderr[-500:])

# Add both repos to path
for p in [str(OUR_DIR), str(ORIG_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(str(OUR_DIR))
print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 3: Download Gym288 ───────────────────────────────────
from huggingface_hub import snapshot_download

GYM288_DIR = Path('/content/Gym288-skeleton')
GYM99_PKL  = Path('/content/gym99_from_gym288.pkl')
DATA_DIR   = ORIG_DIR / 'data' / 'gym99'
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (GYM288_DIR.exists() and any(GYM288_DIR.rglob('*.pkl'))):
    print('Downloading Gym288-skeleton...')
    GYM288_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id='Lozumi/Gym288-skeleton', repo_type='dataset',
                      local_dir=str(GYM288_DIR), local_dir_use_symlinks=False)

GYM288_PKL = str(sorted(GYM288_DIR.rglob('*.pkl'))[0])
print('Gym288 pickle:', GYM288_PKL)

In [ ]:
# ── Cell 4: Build Gym99 ───────────────────────────────────────
from src.gym99_builder import build_gym99_from_gym288_pickle

if not GYM99_PKL.exists():
    print('Building Gym99 from Gym288...')
    stats = build_gym99_from_gym288_pickle(
        gym288_dataset_path=GYM288_PKL,
        gym99_dataset_path=str(GYM99_PKL),
    )
    print('Build stats:', stats)
else:
    print('Gym99 pickle already exists.')
print('Gym99 pickle:', GYM99_PKL)

In [ ]:
# ── Cell 5: Load Gym99 tensors (our pipeline) ─────────────────
import numpy as np
from src.gym99_dataset import build_gym99_data_tensors, infer_num_gym99_classes
from src.config import GYM99_NUM_CLASSES

data, labels, flags, _, _ = build_gym99_data_tensors(
    dataset_path=str(GYM99_PKL),
    joint_spec_name='coco18',
    split='all',
    keep_unknown_split=False,
)
# data: (N, 2, T, 18, 1)  float32

train_mask = flags == 1
val_mask   = flags == 0

X_train, y_train = data[train_mask], labels[train_mask]
X_val,   y_val   = data[val_mask],   labels[val_mask]

num_classes = infer_num_gym99_classes(str(GYM99_PKL), fallback=GYM99_NUM_CLASSES)

print(f'Loaded  train={len(X_train)}  val={len(X_val)}  num_classes={num_classes}')
print(f'Tensor  shape={data.shape}  dtype={data.dtype}')

In [ ]:
# ── Cell 6: Export to original ST-GCN format ─────────────────
# Original expects: (N, 3, T, V, M)
# Our data:         (N, 2, T, 18, 1)
# → add confidence channel (ones) to get (N, 3, T, 18, 1)
import pickle

def to_stgcn_format(X):
    N, C, T, V, M = X.shape
    conf = np.ones((N, 1, T, V, M), dtype=np.float32)
    return np.concatenate([X, conf], axis=1)  # (N, 3, T, V, M)

train_data = to_stgcn_format(X_train)  # (N_train, 3, T, 18, 1)
val_data   = to_stgcn_format(X_val)    # (N_val,   3, T, 18, 1)

# Label format: (list_of_sample_names, list_of_labels)
train_label = ([f'train_{i}' for i in range(len(y_train))], y_train.tolist())
val_label   = ([f'val_{i}'   for i in range(len(y_val))],   y_val.tolist())

np.save(str(DATA_DIR / 'train_data.npy'), train_data)
np.save(str(DATA_DIR / 'val_data.npy'),   val_data)

with open(DATA_DIR / 'train_label.pkl', 'wb') as f:
    pickle.dump(train_label, f)
with open(DATA_DIR / 'val_label.pkl', 'wb') as f:
    pickle.dump(val_label, f)

print(f'train_data.npy : {train_data.shape}')
print(f'val_data.npy   : {val_data.shape}')
print(f'Labels saved to {DATA_DIR}')

In [ ]:
# ── Cell 7: Write Gym99 config YAML ──────────────────────────
# Based on the original kinetics-skeleton train config.
# Joint ordering note: our data is COCO17+center; original 'openpose' layout
# uses a different ordering (neck at joint 1). Graph edges are approximate.

import yaml

CONFIG_DIR = ORIG_DIR / 'config' / 'st_gcn' / 'gym99'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

WORK_DIR  = '/content/work_dir/gym99'
NUM_EPOCH = 90
BATCH     = 64
BASE_LR   = 0.01

train_cfg = {
    'work_dir': WORK_DIR,

    'feeder': 'feeder.feeder_kinetics.Feeder_kinetics',
    'train_feeder_args': {
        'data_path':    str(DATA_DIR / 'train_data.npy'),
        'label_path':   str(DATA_DIR / 'train_label.pkl'),
        'window_size':  int(data.shape[2]),
        'random_choose': True,
        'random_move':   True,
        'mmap':          True,
    },
    'test_feeder_args': {
        'data_path':  str(DATA_DIR / 'val_data.npy'),
        'label_path': str(DATA_DIR / 'val_label.pkl'),
        'window_size': int(data.shape[2]),
        'mmap':        True,
    },

    'model': 'net.st_gcn.Model',
    'model_args': {
        'in_channels': 3,
        'num_class':   int(num_classes),
        'edge_importance_weighting': True,
        'graph_args': {
            'layout':   'openpose',
            'strategy': 'spatial',
        },
    },

    'batch_size':      BATCH,
    'test_batch_size': BATCH,
    'num_epoch':       NUM_EPOCH,
    'optimizer':       'SGD',
    'base_lr':         BASE_LR,
    'weight_decay':    0.0001,
    'step':            [30, 60],
    'num_worker':      0,
    'device':          [0],
    'log_interval':    100,
    'save_interval':   10,
    'save_epoch':      10,
}

cfg_path = CONFIG_DIR / 'train.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(train_cfg, f, default_flow_style=False)

print('Config written to:', cfg_path)
print(yaml.dump(train_cfg, default_flow_style=False))

In [ ]:
# ── Cell 8: Run training ──────────────────────────────────────
import time

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
log_path = Path(WORK_DIR) / 'train.log'

cmd = [
    sys.executable, 'main.py', 'recognition',
    '-c', str(cfg_path),
]

print('Command:', ' '.join(cmd))
print(f'Log: {log_path}')
print()

t0 = time.time()
result = subprocess.run(
    cmd,
    cwd=str(ORIG_DIR),
    stdout=open(log_path, 'w'),
    stderr=subprocess.STDOUT,
)
elapsed = time.time() - t0
h, m = divmod(int(elapsed), 3600)
m, s = divmod(m, 60)
print(f'Finished in {h}h {m}m {s}s  (exit code {result.returncode})')

# Print last 30 lines of log
lines = log_path.read_text().splitlines()
print('\n--- Last 30 log lines ---')
print('\n'.join(lines[-30:]))

In [ ]:
# ── Cell 9: Parse log and plot ────────────────────────────────
import re
import matplotlib.pyplot as plt

log_text = log_path.read_text()

# Original ST-GCN logs lines like:
# [ train][epoch: 1/90] loss: 4.1234  acc: 0.0100
# [ eval ][epoch: 1/90] loss: 4.2000  acc: 0.0200

train_loss, train_acc = [], []
val_loss,   val_acc   = [], []

for line in log_text.splitlines():
    # Try common log patterns from the original repo
    m = re.search(r'\[ *train\].*epoch.*?(\d+).*loss[: ]+([\d.]+).*acc[: ]+([\d.]+)', line, re.I)
    if m:
        train_loss.append(float(m.group(2)))
        train_acc.append(float(m.group(3)))
        continue
    m = re.search(r'\[ *eval\].*epoch.*?(\d+).*loss[: ]+([\d.]+).*acc[: ]+([\d.]+)', line, re.I)
    if m:
        val_loss.append(float(m.group(2)))
        val_acc.append(float(m.group(3)))

if not train_loss:
    print('Could not parse training metrics from log.')
    print('Check log format — first 50 lines:')
    print('\n'.join(log_text.splitlines()[:50]))
else:
    epochs_tr = range(1, len(train_loss) + 1)
    epochs_va = range(1, len(val_loss)   + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs_tr, train_loss, label='train', color='steelblue')
    if val_loss:
        axes[0].plot(epochs_va, val_loss, label='val', color='coral')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_tr, train_acc, label='train', color='steelblue')
    if val_acc:
        axes[1].plot(epochs_va, val_acc, label='val', color='coral')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    fig.suptitle(f'Original ST-GCN on Gym99  ({NUM_EPOCH} epochs, SGD lr={BASE_LR})', fontsize=11)
    plt.tight_layout()
    plt.savefig(Path(WORK_DIR) / 'training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f'Final  train_acc={train_acc[-1]:.4f}  val_acc={val_acc[-1]:.4f}' if val_acc
          else f'Final  train_acc={train_acc[-1]:.4f}')